## 문항 2 : in 연산자 타입 가드와 제네릭 인메모리 저장소

### 1. 데이터 모델 준비
- 노트북마다 커널이 따로 실행되므로 문항 1에서 정의한 `User`, `Product` 모델을 다시 선언합니다.

In [ ]:
// 모든 엔티티의 공통 식별자
interface Identifiable {
    id: number; // 고유 식별 번호
}

// 회원 권한을 제한하는 리터럴 유니온 타입
type UserRole = "admin" | "customer" | "guest";

// Identifiable을 상속한 회원 모델
interface User extends Identifiable {
    name: string; // 회원 이름
    email: string; // 이메일 주소
    role: UserRole; // 회원 권한
    isActive: boolean; // 활성 여부
}

// 상품 카테고리를 제한하는 리터럴 유니온 타입
type Category = "electronics" | "clothing" | "books";

// Identifiable을 상속한 상품 모델
interface Product extends Identifiable {
    title: string; // 상품명
    price: number; // 정가
    category: Category; // 카테고리
    stock: number; // 재고 수량
    discountRate?: number; // 할인율(0~1)이며 생략할 수 있다.
}

// 회원 더미 데이터
const users: User[] = [
    { id: 1, name: "김관리", email: "admin@shop.com", role: "admin", isActive: true },
    { id: 2, name: "이고객", email: "lee@shop.com", role: "customer", isActive: true },
];

// 상품 더미 데이터
const products: Product[] = [
    { id: 101, title: "무선 이어폰", price: 129000, category: "electronics", stock: 15, discountRate: 0.1 },
    { id: 104, title: "타입스크립트 교과서", price: 32000, category: "books", stock: 12 },
];

console.log("회원 수:", users.length);
console.log("상품 수:", products.length);

회원 수: 2


상품 수: 2


### 2. in 연산자를 활용한 타입 좁히기 [필수]
- `User | Product` 유니온 타입은 두 타입에 **공통으로 있는 속성(id)만** 바로 접근할 수 있습니다.
- `in` 연산자로 한쪽 타입에만 있는 속성이 있는지 검사하면, 컴파일러가 해당 블록 안에서 타입을 자동으로 좁혀 줍니다.

| **분기** | **검사 조건** | **좁혀진 타입** | **반환 형식** |
| --- | --- | --- | --- |
| 참 | `'email' in item` | `User` | `[회원] {name} ({role})` |
| 거짓 (else) | `email` 속성 없음 | `Product` | `[상품] {title} - {price}원` |

타입 좁히기(Narrowing)
> 조건문으로 컴파일러에게 변수의 타입을 입증해서, 해당 블록 안에서 그 타입의 속성에 안전하게 접근할 수 있게 하는 과정입니다.

#### 2.1 renderItemSummary 함수 작성

In [ ]:
// 회원 또는 상품을 받아 요약 문자열을 만드는 함수
function renderItemSummary(item: User | Product): string {
    if ('email' in item) { // User
        return `[회원] ${item.name} (${item.role})`;
    } else { // Product
        return `[상품] ${item.title} - ${item.price}원`;
    }
}

#### 2.2 실행 결과 확인

In [ ]:
// 회원과 상품이 섞인 목록
const items: (User | Product)[] = [...users, ...products];

for (const item of items) {
    console.log(renderItemSummary(item));
}

[회원] 김관리 (admin)


[회원] 이고객 (customer)


[상품] 무선 이어폰 - 129000원


[상품] 타입스크립트 교과서 - 32000원


### 3. 타입 좁히기 전 접근 확인
- 타입을 좁히지 않고 한쪽 타입에만 있는 속성에 접근하면 **컴파일 단계에서 차단**됩니다.
- 아래 셀은 의도적으로 컴파일 에러를 확인하는 셀입니다. (에러 출력이 정상 결과입니다)

| **접근 속성** | **예상 에러** |
| --- | --- |
| `item.name` | Property 'name' does not exist on type 'Product' |
| `item.title` | Property 'title' does not exist on type 'User' |

In [ ]:
// [컴파일 에러] 타입을 좁히기 전에는 고유 속성에 접근할 수 없다.
function wrongSummary(item: User | Product): string {
    return `${item.name} ${item.title}`;
}

3:20 - Property 'name' does not exist on type 'User | Product'.


3:20 - Property 'name' does not exist on type 'Product'.


3:33 - Property 'title' does not exist on type 'User | Product'.


3:33 - Property 'title' does not exist on type 'User'.
